In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface


In [14]:
recording_raw = se.read_intan(f"/media/ubuntu/sda/mouse_test/raw_data/20251125-128pi-5#_251125_214012.rhd", stream_id= '0')

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

In [15]:
# 读取probe配置文件
probe_df = pd.read_csv('/media/ubuntu/sda/mouse_test/probe/probe_128channel.csv')

# 定义函数：将数字转换为'C-xxx'格式
def num_to_channel_id(num):
    return f'C-{int(num):03d}'

# 处理4组channel_id
probe_data = {}
for i in range(1, 5):
    col_name = f'channel_id_{i}'
    # 获取该组的channel_id（数字）
    channel_nums = probe_df[col_name].dropna().values
    # 转换为'C-xxx'格式
    channel_ids_c = [num_to_channel_id(num) for num in channel_nums]
    # 按照数字大小排序（先提取数字，排序，再转换回'C-xxx'格式）
    channel_nums_sorted = sorted(channel_nums)
    channel_ids_sorted = [num_to_channel_id(num) for num in channel_nums_sorted]
    
    # 获取对应的x, y坐标（需要按照排序后的channel_id顺序）
    # 创建一个字典，将channel_id映射到(x, y)
    channel_to_xy = {}
    for idx, row in probe_df.iterrows():
        if pd.notna(row[col_name]):
            ch_id = num_to_channel_id(row[col_name])
            channel_to_xy[ch_id] = (row['x'], row['y'])
    
    # 按照排序后的channel_id顺序获取坐标
    positions = [channel_to_xy[ch_id] for ch_id in channel_ids_sorted]
    
    probe_data[f'probe{i}'] = {
        'channel_ids': channel_ids_sorted,
        'positions': positions,
        'channel_nums': channel_nums_sorted
    }
    
    print(f"Probe {i}: {len(channel_ids_sorted)} channels")
    print(f"  Channel IDs (sorted): {channel_ids_sorted[:5]}...{channel_ids_sorted[-5:]}")
    print(f"  First few positions: {positions[:3]}")

# 构建probe对象
probes = {}
for i in range(1, 5):
    probe_name = f'probe{i}'
    data = probe_data[probe_name]
    
    # 创建Probe对象
    probe = probeinterface.Probe(ndim=2)
    probe.set_contacts(
        positions=data['positions'],
        shapes='circle',
        contact_ids=data['channel_ids'],
        shape_params={'radius': 20}  # 单位：um
    )
    
    # 设置device_channel_indices（使用数字索引0到31）
    device_channel_indices = list(range(len(data['channel_ids'])))
    probe.set_device_channel_indices(device_channel_indices)
    
    # 添加可视化标识
    probe.annotate(contact_ids=data['channel_ids'])
    
    probes[probe_name] = probe
    print(f"\n{probe_name} created: {len(data['channel_ids'])} channels")

# 获取每个probe的channel_ids（用于后续拆分recording）
probe_1_channel_ids = probe_data['probe1']['channel_ids']
probe_2_channel_ids = probe_data['probe2']['channel_ids']
probe_3_channel_ids = probe_data['probe3']['channel_ids']
probe_4_channel_ids = probe_data['probe4']['channel_ids']


Probe 1: 32 channels
  Channel IDs (sorted): ['C-008', 'C-009', 'C-010', 'C-011', 'C-014']...['C-056', 'C-057', 'C-058', 'C-059', 'C-061']
  First few positions: [(np.int64(99), np.int64(660)), (np.int64(138), np.int64(100)), (np.int64(2), np.int64(100))]
Probe 2: 32 channels
  Channel IDs (sorted): ['C-000', 'C-001', 'C-002', 'C-003', 'C-004']...['C-049', 'C-054', 'C-060', 'C-062', 'C-063']
  First few positions: [(np.int64(136), np.int64(50)), (np.int64(140), np.int64(0)), (np.int64(95), np.int64(710))]
Probe 3: 32 channels
  Channel IDs (sorted): ['C-064', 'C-065', 'C-067', 'C-073', 'C-078']...['C-123', 'C-124', 'C-125', 'C-126', 'C-127']
  First few positions: [(np.int64(9), np.int64(260)), (np.int64(111), np.int64(510)), (np.int64(29), np.int64(510))]
Probe 4: 32 channels
  Channel IDs (sorted): ['C-066', 'C-068', 'C-069', 'C-070', 'C-071']...['C-113', 'C-116', 'C-117', 'C-118', 'C-119']
  First few positions: [(np.int64(21), np.int64(410)), (np.int64(123), np.int64(360)), (np.int

In [16]:
# 拆分 recording_f 为 4 个小的 recording
recording_f_probe1 = recording_f.select_channels(channel_ids=probe_1_channel_ids)
recording_f_probe2 = recording_f.select_channels(channel_ids=probe_2_channel_ids)
recording_f_probe3 = recording_f.select_channels(channel_ids=probe_3_channel_ids)
recording_f_probe4 = recording_f.select_channels(channel_ids=probe_4_channel_ids)

print(f"Recording probe 1: {recording_f_probe1.get_num_channels()} channels")
print(f"Recording probe 2: {recording_f_probe2.get_num_channels()} channels")
print(f"Recording probe 3: {recording_f_probe3.get_num_channels()} channels")
print(f"Recording probe 4: {recording_f_probe4.get_num_channels()} channels")

# 为每个recording添加对应的probe对象
recording_f_probe1 = recording_f_probe1.set_probe(probes['probe1'], group_mode='by_probe')
recording_f_probe2 = recording_f_probe2.set_probe(probes['probe2'], group_mode='by_probe')
recording_f_probe3 = recording_f_probe3.set_probe(probes['probe3'], group_mode='by_probe')
recording_f_probe4 = recording_f_probe4.set_probe(probes['probe4'], group_mode='by_probe')

print("\nProbes added to recordings:")
print(f"  Probe 1 channel order: {recording_f_probe1.get_channel_ids()[:5]}...{recording_f_probe1.get_channel_ids()[-5:]}")
print(f"  Probe 2 channel order: {recording_f_probe2.get_channel_ids()[:5]}...{recording_f_probe2.get_channel_ids()[-5:]}")
print(f"  Probe 3 channel order: {recording_f_probe3.get_channel_ids()[:5]}...{recording_f_probe3.get_channel_ids()[-5:]}")
print(f"  Probe 4 channel order: {recording_f_probe4.get_channel_ids()[:5]}...{recording_f_probe4.get_channel_ids()[-5:]}")

Recording probe 1: 32 channels
Recording probe 2: 32 channels
Recording probe 3: 32 channels
Recording probe 4: 32 channels

Probes added to recordings:
  Probe 1 channel order: ['C-008' 'C-009' 'C-010' 'C-011' 'C-014']...['C-056' 'C-057' 'C-058' 'C-059' 'C-061']
  Probe 2 channel order: ['C-000' 'C-001' 'C-002' 'C-003' 'C-004']...['C-049' 'C-054' 'C-060' 'C-062' 'C-063']
  Probe 3 channel order: ['C-064' 'C-065' 'C-067' 'C-073' 'C-078']...['C-123' 'C-124' 'C-125' 'C-126' 'C-127']
  Probe 4 channel order: ['C-066' 'C-068' 'C-069' 'C-070' 'C-071']...['C-113' 'C-116' 'C-117' 'C-118' 'C-119']


In [ ]:
# 导入导出模块

# 对每个 probe 的 recording 进行 spike sorting
probe_recordings = {
    'probe1': recording_f_probe1,
    'probe2': recording_f_probe2,
    'probe3': recording_f_probe3,
    'probe4': recording_f_probe4
}

base_output_folder = '/media/ubuntu/sda/mouse_test/sorted/20251125-128pi-5_251125_214012'
os.makedirs(base_output_folder, exist_ok=True)
for probe_name, recording_probe in probe_recordings.items():
    print(f"\n{'='*60}")
    print(f"Processing {probe_name}...")
    print(f"{'='*60}")
    
    # 保存预处理后的 recording
    output_folder = f'{base_output_folder}/{probe_name}'
    recording_preprocessed = recording_probe.save(format="binary")
    print(f"Saved preprocessed recording: {recording_preprocessed}")
    
    # 运行 Kilosort4
    print(f"Running Kilosort4 for {probe_name}...")
    sorting_kilosort4 = ss.run_sorter(
        sorter_name="kilosort4", 
        recording=recording_preprocessed, 
        folder=output_folder + "/kilosort4"
    )
    
    # 创建 SortingAnalyzer
    print(f"Creating SortingAnalyzer for {probe_name}...")
    analyzer_kilosort4 = si.create_sorting_analyzer(
        sorting=sorting_kilosort4, 
        recording=recording_preprocessed, 
        format='binary_folder', 
        folder=output_folder + '/analyzer_kilosort4_binary'
    )
    
    # 计算扩展
    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "noise_levels",
        "templates",
        "spike_amplitudes",
        "unit_locations",
        "spike_locations",
        "correlograms",
        "template_similarity"
    ]
    
    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "spike_locations": {"ms_before": 0.1},
        "correlograms": {"bin_ms": 0.1},
        "template_similarity": {"method": "cosine_similarity"}
    }
    
    print(f"Computing extensions for {probe_name}...")
    analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)
    
    # 计算质量指标
    print(f"Computing quality metrics for {probe_name}...")
    qm_params = sqm.get_default_qm_params()
    analyzer_kilosort4.compute("quality_metrics", qm_params)
    

    sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

print(f"\n{'='*60}")
print("All probes processed successfully!")
print(f"{'='*60}")



Processing probe1...
Use cache_folder=/tmp/spikeinterface_cache/tmp74rvrlx1/SUZQ5268
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=20,000 - chunk_memory=1.22 MiB - total_memory=1.22 MiB - chunk_duration=1.00s


write_binary_recording (no parallelization):   0%|          | 0/622 [00:00<?, ?it/s]

Saved preprocessed recording: BinaryFolderRecording: 32 channels - 20.0kHz - 1 segments - 12,431,744 samples 
                       621.59s (10.36 minutes) - int16 dtype - 758.77 MiB
Running Kilosort4 for probe1...


100%|██████████| 2/2 [00:05<00:00,  2.64s/it]

Creating SortingAnalyzer for probe1...


...

Run:
phy template-gui  /media/ubuntu/sda/mouse_test/sorted/20251125-128pi-5_251125_214012/probe1/phy_folder_for_kilosort/params.py

Processing probe2...
Use cache_folder=/tmp/spikeinterface_cache/tmpgvyrow11/88GDXIEI
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=20,000 - chunk_memory=1.22 MiB - total_memory=1.22 MiB - chunk_duration=1.00s


write_binary_recording (no parallelization):   0%|          | 0/622 [00:00<?, ?it/s]

Saved preprocessed recording: BinaryFolderRecording: 32 channels - 20.0kHz - 1 segments - 12,431,744 samples 
                       621.59s (10.36 minutes) - int16 dtype - 758.77 MiB
Running Kilosort4 for probe2...


100%|██████████| 208/208 [00:13<00:00, 15.74it/s]


SpikeSortingError: Spike sorting error trace:
Traceback (most recent call last):
  File "/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/sorters/basesorter.py", line 270, in run_from_folder
    SorterClass._run_from_folder(sorter_output_folder, sorter_params, verbose)
  File "/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/sorters/external/kilosort4.py", line 380, in _run_from_folder
    st, tF, _, _ = detect_spikes(**detect_spikes_kwargs)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/kilosort/run_kilosort.py", line 768, in detect_spikes
    st0, tF, ops = spikedetect.run(
                   ^^^^^^^^^^^^^^^^
  File "/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/kilosort/spikedetect.py", line 205, in run
    ops['wPCA'], ops['wTEMP'] = extract_wPCA_wTEMP(
                                ^^^^^^^^^^^^^^^^^^^
  Fi

In [18]:
recording_raw = se.read_intan(f"/media/ubuntu/sda/mouse_test/raw_data/20251125-128pi-1#_251125_195122.rhd", stream_id= '0')

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

In [21]:
probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
recording_f = recording_f.set_probegroup(probe)

In [24]:
output_folder = f'/media/ubuntu/sda/mouse_test/raw_data/20251125-128pi-1_251125_195122_test'
recording_preprocessed = recording_f.save(format="binary")
print(f"Saved preprocessed recording: {recording_preprocessed}")

# 运行 Kilosort4
print(f"Running Kilosort4 for {probe_name}...")
sorting_kilosort4 = ss.run_sorter(
    sorter_name="kilosort4", 
    recording=recording_preprocessed, 
    folder=output_folder + "/kilosort4"
)

# 创建 SortingAnalyzer
print(f"Creating SortingAnalyzer for {probe_name}...")
analyzer_kilosort4 = si.create_sorting_analyzer(
    sorting=sorting_kilosort4, 
    recording=recording_preprocessed, 
    format='binary_folder', 
    folder=output_folder + '/analyzer_kilosort4_binary'
)

# 计算扩展
extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "spike_amplitudes",
    "unit_locations",
    "spike_locations",
    "correlograms",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

print(f"Computing extensions for {probe_name}...")
analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)

# 计算质量指标
print(f"Computing quality metrics for {probe_name}...")
qm_params = sqm.get_default_qm_params()
analyzer_kilosort4.compute("quality_metrics", qm_params)


sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

Use cache_folder=/tmp/spikeinterface_cache/tmpzou7zm9p/9WM009BY
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=20,000 - chunk_memory=4.88 MiB - total_memory=4.88 MiB - chunk_duration=1.00s


write_binary_recording (no parallelization):   0%|          | 0/432 [00:00<?, ?it/s]

Saved preprocessed recording: BinaryFolderRecording: 128 channels - 20.0kHz - 1 segments - 8,628,864 samples 
                       431.44s (7.19 minutes) - int16 dtype - 2.06 GiB
Running Kilosort4 for probe2...


100%|██████████| 8/8 [01:00<00:00,  7.51s/it]


Creating SortingAnalyzer for probe2...


...

write_binary_recording (no parallelization):   0%|          | 0/432 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/131 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/131 [00:00<?, ?it/s]

extract PCs (no parallelization):   0%|          | 0/432 [00:00<?, ?it/s]

Run:
phy template-gui  /media/ubuntu/sda/mouse_test/raw_data/20251125-128pi-1_251125_195122_test/phy_folder_for_kilosort/params.py
